In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms

import matplotlib.pyplot as plt

import gc
import random

import os
import glob
from dotenv import load_dotenv

from dataset_classes.lazy_dataset import LazyDataset
from src.setup_model import setup_model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device is {device}")

In [ ]:
load_dotenv("config.env")

# --- Dataset parameters ---
RANDOM_SEED = int(os.getenv("RANDOM_SEED"))
USE_DVS = int(os.getenv("USE_DVS"))
USE_APS = int(os.getenv("USE_APS"))
MODEL_DIMS = USE_DVS + USE_APS
if MODEL_DIMS == 0: raise Exception("MODEL_DIMS must be non-zero")

BINSIZE_MS = int(os.getenv("BINSIZE_MS"))
HEIGHT = int(os.getenv("HEIGHT"))
WIDTH = int(os.getenv("WIDTH"))
PRECISION = int(os.getenv("PRECISION"))
DTYPE = np.float16 if PRECISION == 16 else np.float32

APS_FRAME_NAME = f"aps_frame_{HEIGHT}x{WIDTH}"
DVS_FRAME_NAME = f"dvs_frame_{HEIGHT}x{WIDTH}"

# --- Location parameters ---
DATASET_DIR = os.getenv("OUT_DIR")
MODEL_SAVE_DIR = os.getenv("MODEL_SAVE_DIR")
# --- Preprocessing parameters ---
PERFORM_PRUNING = bool(int(os.getenv("PERFORM_PRUNING")))
# km/h for speed to be considered high enough
SPEED_THRESHOLD = float(os.getenv("SPEED_THRESHOLD"))
# absolute value for degree of steering to be considered high enough
STEERING_THRESHOLD = float(os.getenv("STEERING_THRESHOLD"))
STEERING_DISCARD_CHANCE = float(os.getenv("STEERING_DISCARD_CHANCE"))

TRAINING_DIR = f"{DATASET_DIR}/Training"
TEST_DIR = f"{DATASET_DIR}/Test"

# --- Training parameters ---
MODEL_TYPE = os.getenv("MODEL_TYPE")
PRETRAINED = bool(int(os.getenv("PRETRAINED")))
WARM_START= bool(int(os.getenv("WARM_START")))
CLEAN_OLD_MODELS = bool(int(os.getenv("CLEAN_OLD_MODELS")))
if WARM_START and CLEAN_OLD_MODELS: raise Exception("Can't have WARM_START and CLEAN_OLD_MODELS at the same time!")
TRAINING_SPLIT = float(os.getenv("TRAINING_SPLIT"))
EPOCHS = int(os.getenv("EPOCHS"))
BATCH_SIZE = int(os.getenv("BATCH_SIZE"))
WINDOW_SIZE = int(os.getenv("WINDOW_SIZE"))
LEARNING_RATE = float(os.getenv("LEARNING_RATE"))
WEIGHT_DECAY = float(os.getenv("WEIGHT_DECAY"))

# --- Fix randomness ---
random.seed(RANDOM_SEED)


In [ ]:
# SETUP MODEL
model = setup_model(MODEL_TYPE, MODEL_DIMS, PRETRAINED, device)

In [ ]:
inv_apsdvs_max = 1 / 255

# These normalisation functions are not standardised intentionally, to take advantage of pre-computing with some known values.
# used to normalise the aps and dvs frames to the range [0, 1]
def normalise_dvsaps(data):
    # shorthand for (data - self.apsdvs_min) / (self.apsdvs_max - self.apsdvs_min) since we know the min is 0
    new_data = np.empty_like(data, dtype=DTYPE)
    new_data = data * inv_apsdvs_max
    return new_data

transform = transforms.Compose([
    lambda x: normalise_dvsaps(x),
    transforms.ToPILImage(),
    transforms.ToTensor(),
])

# normalise the steering values from -180 to 180 to -1 to 1
# Precompute normalization coefficients
steering_slope = 1 / 180  # (output_max - output_min)/(input_max - input_min) = (1-(-1))/(180-(-180))
def normalise_steering(data):
    return data * steering_slope

# used for getting the data out at the end.
inv_steering_slope = 180
def denormalise_steering(data):
    return data * inv_steering_slope

target_transform = transforms.Compose([
    lambda x: normalise_steering(x)
])

train_dataset = LazyDataset(TRAINING_DIR, transform=transform, target_transform=target_transform, window_size=WINDOW_SIZE, use_aps=USE_APS, use_dvs=USE_DVS, DVS_FRAME_NAME=DVS_FRAME_NAME, APS_FRAME_NAME=APS_FRAME_NAME, DTYPE=DTYPE)
test_dataset = LazyDataset(TEST_DIR, transform=transform, target_transform=target_transform, window_size=WINDOW_SIZE, use_aps=USE_APS, use_dvs=USE_DVS, DVS_FRAME_NAME=DVS_FRAME_NAME, APS_FRAME_NAME=APS_FRAME_NAME, DTYPE=DTYPE)

# load the data, can implement batching etc. here
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=True, num_workers=4, pin_memory=True, prefetch_factor=5)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=True, num_workers=4, pin_memory=True, prefetch_factor=5)
print(f"Train loader has {len(train_loader)} batches of length {WINDOW_SIZE}, equalling {len(train_loader) * BATCH_SIZE * WINDOW_SIZE} frames.\n-Test loader has {len(test_loader)} batches of length {WINDOW_SIZE}, equalling {len(test_loader) * BATCH_SIZE * WINDOW_SIZE} frames.")

In [ ]:
def warm_start(model, optimizer):
        """
        Load the last model and optimizer state from the last saved model, returns starting epoch and best loss
        """
        last_model_file = glob.glob(f"{MODEL_SAVE_DIR}/last/*")[-1]
        load_dict = torch.load(last_model_file, map_location=device)
        model.load_state_dict(load_dict["model_state_dict"])
        optimizer.load_state_dict(load_dict["optimizer_state_dict"])
        epoch = load_dict["epoch"]
        best_test_loss = load_dict["best_test_loss"]
        return epoch + 1, best_test_loss

In [ ]:
starting_epoch = 0
best_test_loss = np.inf

#Loss and optimizer
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

if WARM_START:
    starting_epoch, best_test_loss = warm_start(model, optimizer)

In [ ]:
def clean_models(directory):
    '''
    Delete the models in a given directory
    '''
    for model_file in glob.glob(directory + "/*.pth"):
        os.remove(model_file)

In [ ]:
def save_model(directory, model, optimizer, epoch, loss, best_test_loss, *, clean_old=False):
    '''
    Saves the model in the given location.
    If clean_old, cleans the old model files in the directory
    '''
    state = {
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'epoch': epoch,
                'loss': loss,
                'best_test_loss': best_test_loss
    }
    
    if clean_old:
        clean_models(directory)
    
    torch.save(state, f"{directory}/e{epoch + 1}_L{loss:.2f}.pth")

In [ ]:
if CLEAN_OLD_MODELS:
  clean_models(MODEL_SAVE_DIR)

train_losses = []
test_losses = []

scaler = torch.amp.GradScaler("cuda")
for epoch in range(starting_epoch, EPOCHS):
    epoch_train_loss = 0
    for combined_images, steering_angle in train_loader:
      if MODEL_TYPE == "Resnet":
        # Squeeze the sequence dimension
        combined_images = combined_images.squeeze(1)

      optimizer.zero_grad()
      combined_images = combined_images.to(device)
      steering_angle = steering_angle.to(device)
      with torch.autocast(device_type="cuda", dtype=torch.float16):
        outputs = model(combined_images).squeeze(2)
        loss = criterion(outputs, steering_angle)

      scaler.scale(loss).backward()
      scaler.step(optimizer)
      scaler.update()
      epoch_train_loss += loss.item()
      del combined_images, steering_angle, outputs
      torch.cuda.empty_cache()
      gc.collect()

    # Average training loss for the epoch
    epoch_train_loss /= len(train_loader)
    train_losses.append(epoch_train_loss)

    #Validation
    epoch_test_loss = 0
    with torch.no_grad():
          for combined_images, steering_angle in test_loader:
              if MODEL_TYPE == "Resnet":
                # Squeeze the sequence dimension
                combined_images = combined_images.squeeze(1)

              combined_images = combined_images.to(device)
              steering_angle = steering_angle.to(device)
              with torch.autocast(device_type="cuda", dtype=torch.float16):
                outputs = model(combined_images).squeeze(2)
                loss = criterion(outputs, steering_angle)
              epoch_test_loss += loss.item()
              del combined_images, steering_angle, outputs
          epoch_test_loss /= len(test_loader)
          
          # Save the best
          if (epoch_test_loss < best_test_loss):
              best_test_loss = epoch_test_loss
              save_model(f"{MODEL_SAVE_DIR}/best", model, optimizer, epoch, epoch_test_loss, best_test_loss, clean_old=True)

          # Save every 20 epochs
          if not ((epoch + 1) % 20):
              save_model(MODEL_SAVE_DIR, model, optimizer, epoch, epoch_test_loss, best_test_loss)
          # Save the last model
          save_model(f"{MODEL_SAVE_DIR}/last", model, optimizer, epoch, epoch_test_loss, best_test_loss, clean_old=True)

          # Report on the loss
          test_losses.append(epoch_test_loss)
          print(f"Epoch [{epoch + 1}/{EPOCHS}], Train Loss: {epoch_train_loss: .4f} Test Loss: {epoch_test_loss: .4f}")
            
# Plotting the training and validation loss
plt.figure(figsize=(10, 5))
plt.plot(range(1, EPOCHS + 1), train_losses, label='Training Loss')
plt.plot(range(1, EPOCHS + 1), test_losses, label='Test Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss (Degrees of Steering)')
plt.title('Training and Test Losses')
plt.legend()
plt.show()